# RAG Ingestion with Ray Data

Ingest PDFs into Milvus for RAG using a 3-stage Ray Data streaming pipeline:
**Docling** (parse + chunk) → **Embed** → **Milvus** (store).

## Embedding Modes

This pipeline supports **two modes** for generating embeddings:

**Mode 1: Local** (CPU)
- Embedding model runs inside the pipeline via `sentence-transformers`
- No GPU required — simpler setup, good for testing

**Mode 2: Service** (GPU, default)
- Embedding via `vLLMEngineProcessorConfig` on GPU workers
- Better throughput for production workloads

Toggle between modes by setting `EMBEDDING_MODE = "local"` or `"service"` in the
Configure cell below.

---

The pipeline runs as a **RayJob** on a **RayCluster** created via
`codeflare-sdk` (see **Create RayCluster** below).
Ray Data streams the stages so they overlap — embedding starts before all PDFs
finish parsing. The pipeline code lives in `docling_milvus_process.py`.

### Prerequisites

- **Red Hat OpenShift AI** with the KubeRay operator installed
- **Milvus** accessible from the Ray cluster
- PDFs uploaded to a **ReadWriteMany PVC** mounted on all Ray nodes
- A workbench with `codeflare-sdk` installed (this notebook)

In [ ]:
%pip install -q \
    "codeflare-sdk>=0.28.0,<1.0" \
    "pymilvus>=2.4.0,<3.0" \
    "ray[data]>=2.54.0,<3.0" \
    "sentence-transformers>=3.0.0,<4.0" \
    "docling>=2.0.0,<3.0"

In [ ]:
import json as _json_mod
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

from codeflare_sdk import Cluster, ClusterConfiguration, RayJob

## Authenticate

Log in to your OpenShift cluster. Replace the command below with your own
`oc login` invocation (token + server URL), or skip this cell if you are
already authenticated via the workbench.

In [ ]:
# Paste your own oc login command here:
# !oc login --token=<your-token> --server=<your-api-url>

!oc whoami

## Configure

> ⚠️ **You MUST update the "Required" values below to match your cluster.**

In [ ]:
# ============================================================
# REQUIRED — update these for your cluster
# ============================================================
CLUSTER_NAME = "raytest"
NAMESPACE = "rag-example"
PVC_CLAIM_NAME = "rag-data-pvc"  # ReadWriteMany; must exist in NAMESPACE
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"

# ⚠️ WARNING: When True, the pipeline DROPS and recreates the Milvus collection, destroying all existing data.
# Set to False to fail gracefully if the collection already exists.
DROP_EXISTING_COLLECTION = "true"

# ============================================================
# EMBEDDING MODE
# ============================================================
# "local"   — sentence-transformers on CPU; simpler setup, good for testing
# "service" — vLLM via vLLMEngineProcessorConfig on GPU; better throughput for production
EMBEDDING_MODE = "service"
EMBEDDING_MODEL = "ibm-granite/granite-embedding-125m-english"
EMBEDDING_DIM = "768"

# ============================================================
# OPTIONAL — safe defaults, tune for your hardware
# ============================================================

# Storage (PVC mounted on all Ray workers)
PVC_MOUNT_PATH = "/mnt/data"
INPUT_PATH = "input/pdfs"

# Milvus
MILVUS_PORT = "19530"
MILVUS_DB = "default"
MILVUS_COLLECTION = "rag_documents"
MILVUS_TEXT_MAX_CHARS = "8192"

# Local embedding (only used when EMBEDDING_MODE = "local")
EMBEDDING_BATCH_SIZE = "32"
NUM_EMBEDDING_ACTORS = "2"

# vLLM embedding (only used when EMBEDDING_MODE = "service")
VLLM_MODEL_SOURCE = EMBEDDING_MODEL
VLLM_BATCH_SIZE = "4"
VLLM_CONCURRENCY = "1"
VLLM_ENGINE_KWARGS_JSON = '{"enforce_eager":true,"gpu_memory_utilization":0.8}'
VLLM_ACCELERATOR_TYPE = ""  # optional Ray accelerator type for GPU workers

# Pipeline tuning
NUM_FILES = "0"  # PDFs to process (0 = all)
# 70% rule: max_actors ≈ floor(total_cluster_cpus × 0.70 / CPUS_PER_ACTOR)
NUM_ACTORS = "6"  # Docling parsing actors — see README "Sizing for your hardware"
CPUS_PER_ACTOR = "4"  # CPUs per Docling actor
NUM_MILVUS_ACTORS = "2"
BATCH_SIZE = "2"  # PDFs per actor batch
MILVUS_BATCH_SIZE = "64"
REPARTITION_FACTOR = "2"
CHUNK_MAX_TOKENS = "256"

print(f"Cluster:    {CLUSTER_NAME} in {NAMESPACE}")
print(f"Input:      {PVC_MOUNT_PATH}/{INPUT_PATH}")
print(f"Milvus:     {MILVUS_HOST}:{MILVUS_PORT}/{MILVUS_DB}.{MILVUS_COLLECTION}")
print(
    f"Embedding:  mode={EMBEDDING_MODE}, model={EMBEDDING_MODEL} (dim={EMBEDDING_DIM})"
)
if EMBEDDING_MODE == "service":
    print(f"  vLLM:     batch_size={VLLM_BATCH_SIZE}, concurrency={VLLM_CONCURRENCY}")
    print(f"  GPU:      {VLLM_ENGINE_KWARGS_JSON}")
else:
    print(
        f"  Local:    batch_size={EMBEDDING_BATCH_SIZE}, actors={NUM_EMBEDDING_ACTORS}"
    )
print(
    f"Actors:     {NUM_ACTORS} docling x {CPUS_PER_ACTOR} CPUs, {NUM_MILVUS_ACTORS} milvus"
)
print(f"Files:      {'ALL' if NUM_FILES == '0' else NUM_FILES}")

## Validate Prerequisites (optional)

Run the cell below to check that your cluster is ready before proceeding.
All checks use `oc` commands — skip this cell if you've already verified your setup.

In [ ]:
def _check(description, fix_hint, cmd):
    result = subprocess.run(cmd, capture_output=True, text=True)
    status = "PASS" if result.returncode == 0 else "FAIL"
    suffix = "" if result.returncode == 0 else f" -- {fix_hint}"
    print(f"[{status}] {description}{suffix}")
    return result.returncode == 0


print(f"=== RAG Pipeline Prerequisite Validation (namespace: {NAMESPACE}) ===\n")
results = [
    _check("oc authenticated", "Run: oc login <cluster-url>", ["oc", "whoami"]),
    _check(
        f"Namespace '{NAMESPACE}' exists",
        f"Create it with: oc new-project {NAMESPACE}",
        ["oc", "get", "namespace", NAMESPACE],
    ),
    _check(
        "KubeRay CRDs installed",
        "Install the KubeRay operator via OperatorHub",
        ["oc", "get", "crd", "rayclusters.ray.io"],
    ),
    _check(
        "RBAC: can create RayJobs",
        f"Request RayJob create permissions in '{NAMESPACE}'",
        ["oc", "auth", "can-i", "create", "rayjobs.ray.io", "-n", NAMESPACE],
    ),
    _check(
        "RBAC: can view pod logs",
        f"Request pods/log get permissions in '{NAMESPACE}'",
        ["oc", "auth", "can-i", "get", "pods/log", "-n", NAMESPACE],
    ),
]
passed = sum(results)
print(f"\n=== {passed}/{len(results)} checks passed ===")
if passed < len(results):
    print("Fix the failures above before proceeding.")

## Download Sample PDFs (optional)

If you don't already have PDFs on your PVC, the cell below downloads the
[Open RAG Benchmark](https://huggingface.co/datasets/deepmatics/open_ragbench)
dataset (1000 arXiv PDFs, CC-BY-NC-4.0). Skip this section if your PVC
already contains input documents.

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.request import urlopen, urlretrieve

PDF_URLS_JSON = (
    "https://huggingface.co/datasets/deepmatics/open_ragbench"
    "/resolve/main/pdf/arxiv/pdf_urls.json"
)
DEST = Path("./sample_pdfs")  # change to PVC path if running on-cluster
MAX_PARALLEL = 8
_SAFE_ID = re.compile(r"^[A-Za-z0-9._-]+$")

DEST.mkdir(parents=True, exist_ok=True)

with urlopen(PDF_URLS_JSON) as resp:
    pdf_urls: dict = _json_mod.loads(resp.read())
print(f"Found {len(pdf_urls)} PDFs in dataset")


def _download(paper_id: str, url: str) -> str:
    if not _SAFE_ID.match(paper_id):
        return f"SKIP  {paper_id} (unsafe id)"
    dest_file = DEST / f"{paper_id}.pdf"
    if dest_file.exists():
        return f"EXISTS {paper_id}"
    try:
        urlretrieve(url, dest_file)
        return f"OK     {paper_id}"
    except Exception as exc:
        dest_file.unlink(missing_ok=True)
        return f"FAIL   {paper_id}: {exc}"


ok = fail = skip = 0
with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as pool:
    futures = {pool.submit(_download, pid, url): pid for pid, url in pdf_urls.items()}
    for i, fut in enumerate(as_completed(futures), 1):
        result = fut.result()
        if result.startswith("OK"):
            ok += 1
        elif result.startswith("FAIL"):
            fail += 1
            print(result)
        else:
            skip += 1
        if i % 100 == 0:
            print(f"  progress: {i}/{len(pdf_urls)}")

actual = len(list(DEST.glob("*.pdf")))
print(
    f"Done. {actual} PDFs in {DEST}/ (downloaded: {ok}, skipped: {skip}, failed: {fail})"
)

## Create RayCluster

Create a RayCluster via `codeflare-sdk`, then patch it with `oc patch` to add
the GPU worker group and tune `rayStartParams` (not exposed by the SDK).

In [ ]:
RAY_IMAGE = "quay.io/kryanbeane/docling:latest"
cluster_config = ClusterConfiguration(
    name=CLUSTER_NAME,
    namespace=NAMESPACE,
    head_cpu_requests=1,
    head_cpu_limits=2,
    head_memory_requests=6,
    head_memory_limits=10,
    num_workers=8,
    worker_cpu_requests=4,
    worker_cpu_limits=4,
    worker_memory_requests=16,
    worker_memory_limits=16,
    image=RAY_IMAGE,
    verify_tls=False,  # OpenShift clusters typically use self-signed certs
    envs={
        "RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION": "0.5",
    },
)

cluster = Cluster(cluster_config)
cluster.up()
cluster.wait_ready()
print(f"Dashboard: {cluster.cluster_dashboard_uri()}")

### Patch RayCluster

The codeflare-sdk doesn't expose `rayStartParams` or additional worker groups.
Patch the cluster to:
- Set head `num-cpus=0` so no actors schedule on the head node
- Set object-store-memory for head and workers
- Add a GPU worker group for vLLM embedding
- Mount the PVC on all nodes

In [ ]:
patch = [
    {
        "op": "replace",
        "path": "/spec/headGroupSpec/rayStartParams/num-cpus",
        "value": "0",
    },
    {
        "op": "add",
        "path": "/spec/headGroupSpec/rayStartParams/object-store-memory",
        "value": "5368709120",
    },
    {
        "op": "add",
        "path": "/spec/headGroupSpec/rayStartParams/disable-usage-stats",
        "value": "true",
    },
    {
        "op": "add",
        "path": "/spec/headGroupSpec/rayStartParams/no-monitor",
        "value": "true",
    },
    # Mount PVC on head
    {
        "op": "add",
        "path": "/spec/headGroupSpec/template/spec/volumes",
        "value": [
            {
                "name": "shared-data",
                "persistentVolumeClaim": {"claimName": PVC_CLAIM_NAME},
            }
        ],
    },
    {
        "op": "add",
        "path": "/spec/headGroupSpec/template/spec/containers/0/volumeMounts",
        "value": [{"mountPath": "/mnt/data", "name": "shared-data"}],
    },
    # Tune CPU workers
    {
        "op": "replace",
        "path": "/spec/workerGroupSpecs/0/rayStartParams/num-cpus",
        "value": "4",
    },
    {
        "op": "add",
        "path": "/spec/workerGroupSpecs/0/rayStartParams/object-store-memory",
        "value": "8589934592",
    },
    # Mount PVC on CPU workers
    {
        "op": "add",
        "path": "/spec/workerGroupSpecs/0/template/spec/volumes",
        "value": [
            {
                "name": "shared-data",
                "persistentVolumeClaim": {"claimName": PVC_CLAIM_NAME},
            }
        ],
    },
    {
        "op": "add",
        "path": "/spec/workerGroupSpecs/0/template/spec/containers/0/volumeMounts",
        "value": [{"mountPath": "/mnt/data", "name": "shared-data"}],
    },
    # Add GPU worker group for vLLM embedding
    {
        "op": "add",
        "path": "/spec/workerGroupSpecs/-",
        "value": {
            "groupName": "gpu-workers",
            "replicas": 1,
            "minReplicas": 1,
            "maxReplicas": 1,
            "rayStartParams": {
                "num-cpus": "2",
                "num-gpus": "1",
                "object-store-memory": "4294967296",
            },
            "template": {
                "metadata": {"labels": {"ray.io/group": "gpu-workers"}},
                "spec": {
                    "containers": [
                        {
                            "name": "ray-gpu-worker",
                            "image": RAY_IMAGE,
                            "imagePullPolicy": "Always",
                            "env": [
                                {
                                    "name": "RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION",
                                    "value": "0.5",
                                }
                            ],
                            "resources": {
                                "limits": {
                                    "cpu": "2",
                                    "memory": "16Gi",
                                    "nvidia.com/gpu": "1",
                                },
                                "requests": {
                                    "cpu": "2",
                                    "memory": "16Gi",
                                    "nvidia.com/gpu": "1",
                                },
                            },
                            "volumeMounts": [
                                {"mountPath": "/mnt/data", "name": "shared-data"}
                            ],
                        }
                    ],
                    "volumes": [
                        {
                            "name": "shared-data",
                            "persistentVolumeClaim": {"claimName": PVC_CLAIM_NAME},
                        }
                    ],
                },
            },
        },
    },
]

patch_json = _json_mod.dumps(patch)
result = subprocess.run(
    [
        "oc",
        "patch",
        "raycluster",
        CLUSTER_NAME,
        "-n",
        NAMESPACE,
        "--type=json",
        f"-p={patch_json}",
    ],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(f"Patch failed: {result.stderr}")
else:
    print(f"Patched RayCluster '{CLUSTER_NAME}' with GPU workers and PVC mounts")

### Pipeline overview

The `_run_pipeline` function chains three stages using Ray Data streaming
execution. Stage 2 varies based on `EMBEDDING_MODE`:

```python
# Stage 1: Parse + chunk (CPU-heavy, bottleneck)
ds = ds.map_batches(
    DoclingChunkActor,
    concurrency=NUM_ACTORS,
    batch_size=BATCH_SIZE,
    num_cpus=CPUS_PER_ACTOR,
)

# Stage 2: Embed
if EMBEDDING_MODE == "local":
    # sentence-transformers on CPU — no GPU required
    ds = ds.map_batches(
        SentenceTransformerEmbedActor,
        concurrency=NUM_EMBEDDING_ACTORS,
        batch_size=EMBEDDING_BATCH_SIZE,
        num_cpus=2,
    )
else:
    # vLLM on GPU via Ray Data LLM processor
    embed_processor = _build_vllm_embed_processor()
    ds = embed_processor(ds)

# Stage 3: Write to Milvus (I/O-bound)
results = ds.map_batches(
    MilvusWriteActor,
    concurrency=NUM_MILVUS_ACTORS,
    batch_size=MILVUS_BATCH_SIZE,
    num_cpus=1,
)
```

Ray Data's streaming executor overlaps stages: downstream actors start as
soon as upstream blocks are ready.

## Submit RayJob

In [ ]:
pipeline_env = {
    "PVC_MOUNT_PATH": PVC_MOUNT_PATH,
    "INPUT_PATH": INPUT_PATH,
    "MILVUS_HOST": MILVUS_HOST,
    "MILVUS_PORT": MILVUS_PORT,
    "MILVUS_DB": MILVUS_DB,
    "MILVUS_COLLECTION": MILVUS_COLLECTION,
    "DROP_EXISTING_COLLECTION": DROP_EXISTING_COLLECTION,
    "MILVUS_TEXT_MAX_CHARS": MILVUS_TEXT_MAX_CHARS,
    "EMBEDDING_MODE": EMBEDDING_MODE,
    "EMBEDDING_MODEL": EMBEDDING_MODEL,
    "EMBEDDING_DIM": EMBEDDING_DIM,
    "EMBEDDING_BATCH_SIZE": EMBEDDING_BATCH_SIZE,
    "NUM_EMBEDDING_ACTORS": NUM_EMBEDDING_ACTORS,
    "VLLM_MODEL_SOURCE": VLLM_MODEL_SOURCE,
    "VLLM_BATCH_SIZE": VLLM_BATCH_SIZE,
    "VLLM_CONCURRENCY": VLLM_CONCURRENCY,
    "VLLM_ENGINE_KWARGS_JSON": VLLM_ENGINE_KWARGS_JSON,
    "VLLM_ACCELERATOR_TYPE": VLLM_ACCELERATOR_TYPE,
    "NUM_FILES": NUM_FILES,
    "NUM_ACTORS": NUM_ACTORS,
    "CPUS_PER_ACTOR": CPUS_PER_ACTOR,
    "NUM_MILVUS_ACTORS": NUM_MILVUS_ACTORS,
    "BATCH_SIZE": BATCH_SIZE,
    "MILVUS_BATCH_SIZE": MILVUS_BATCH_SIZE,
    "REPARTITION_FACTOR": REPARTITION_FACTOR,
    "CHUNK_MAX_TOKENS": CHUNK_MAX_TOKENS,
    "HF_HOME": "/mnt/data/huggingface",
    "XDG_CACHE_HOME": "/tmp/cache",
}

In [ ]:
from datetime import UTC, datetime

job_name = f"rag-ingest-{datetime.now(UTC):%Y%m%d-%H%M%S}"

notebook_dir = Path(".").resolve()
slim_dir = tempfile.mkdtemp(prefix="rag-job-")
shutil.copy2(notebook_dir / "docling_milvus_process.py", slim_dir)
print(f"Working dir: {slim_dir} ({os.listdir(slim_dir)})")

runtime_env = {
    "working_dir": slim_dir,
    "env_vars": pipeline_env,
}
if EMBEDDING_MODE == "local":
    runtime_env["pip"] = ["sentence-transformers>=3.0.0,<4.0"]

job = RayJob(
    job_name=job_name,
    cluster_name=CLUSTER_NAME,
    namespace=NAMESPACE,
    entrypoint="python docling_milvus_process.py",
    runtime_env=runtime_env,
    active_deadline_seconds=10800,
)

job.submit()
print(f"RayJob '{job.name}' submitted.")

## Monitor

In [ ]:
import time as _time
from datetime import UTC
from datetime import datetime as _dt

while True:
    status = job.status()
    print(f"{_dt.now(UTC):%H:%M:%S} — {status}")
    if status in ("SUCCEEDED", "FAILED", "STOPPED"):
        break
    _time.sleep(30)

In [ ]:
!oc get pods -n {NAMESPACE} -l ray.io/cluster={CLUSTER_NAME}

In [ ]:
# Fetch job logs (run after the job completes for the performance report)
!oc logs -l job-name={job.name} -n {NAMESPACE} --tail=100

## Verify

Once the job completes, confirm that vectors were inserted into Milvus.
This connects directly from the workbench — no Ray cluster needed.

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient(uri=f"http://{MILVUS_HOST}:{MILVUS_PORT}", db_name=MILVUS_DB)
client.load_collection(MILVUS_COLLECTION)
stats = client.get_collection_stats(MILVUS_COLLECTION)
row_count = stats.get("row_count", stats.get("data", {}).get("row_count", "?"))
print(f"Collection '{MILVUS_COLLECTION}': {row_count} vectors")
print("\nReady for querying — open rag_query.ipynb to test retrieval.")